# Multiple customer classes in a model

In this example, the model has three customer classes. The customer class determines the arrival distribution and service distribution.

The JSON for this built-in example can be loaded using `json2ciw.datasets.load_three_classes_model`.

## Imports

In [1]:
import json

import ciw
from rich import print

from json2ciw.datasets import load_stroke_pathway_model
from json2ciw.engine import CiwConverter, multiple_replications
from json2ciw.results import summarise_results, summarise_results_by_class, tidy_to_wide_format, tidy_to_wide_format_by_class
from json2ciw.schema import ProcessModel

## Load JSON

In [2]:
json_network = load_stroke_pathway_model()
print(json.dumps(json_network, indent=2))

{
  "name": "Stroke pathway with class-specific routing and reneging",
  "description": "Simple multi-class stroke pathway with class-specific routing, arrival, service, and reneging 
distributions.",
  "customer_classes": [
    {
      "name": "tia",
      "label": "TIA"
    },
    {
      "name": "mild_moderate_stroke",
      "label": "Mild/moderate stroke"
    },
    {
      "name": "severe_stroke",
      "label": "Severe stroke"
    }
  ],
  "activities": [
    {
      "name": "Acute Stroke Unit",
      "type": "activity",
      "resource": {
        "name": "Stroke beds",
        "capacity": 12
      },
      "arrival_distribution": {
        "by_class": {
          "tia": {
            "type": "exponential",
            "parameters": {
              "rate": 2.0
            }
          },
          "mild_moderate_stroke": {
            "type": "exponential",
            "parameters": {
              "rate": 1.2
            }
          },
          "severe_stroke": {
            "type": "exponential",
            "parameters": {
              "rate": 0.6
            }
          }
        }
      },
      "service_distribution": {
        "by_class": {
          "tia": {
            "type": "exponential",
            "parameters": {
              "mean": 2.0
            }
          },
          "mild_moderate_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 4.0
            }
          },
          "severe_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 6.0
            }
          }
        }
      },
      "renege_distribution": {
        "by_class": {
          "tia": {
            "type": "uniform",
            "parameters": {
              "min": 2.0,
              "max": 6.0
            }
          },
          "mild_moderate_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 4.0,
              "max": 10.0
            }
          },
          "severe_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 6.0,
              "max": 12.0
            }
          }
        }
      }
    },
    {
      "name": "Rehab Unit",
      "type": "activity",
      "resource": {
        "name": "Rehab beds",
        "capacity": 8
      },
      "service_distribution": {
        "by_class": {
          "mild_moderate_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 8.0
            }
          },
          "severe_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 14.0
            }
          }
        }
      },
      "renege_distribution": {
        "by_class": {
          "mild_moderate_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 8.0,
              "max": 16.0
            }
          },
          "severe_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 10.0,
              "max": 20.0
            }
          }
        }
      }
    }
  ],
  "transitions": [
    {
      "from": "Acute Stroke Unit",
      "to": "Rehab Unit",
      "probability": {
        "by_class": {
          "tia": 0.0,
          "mild_moderate_stroke": 0.6,
          "severe_stroke": 0.9
        }
      }
    },
    {
      "from": "Acute Stroke Unit",
      "to": "Exit",
      "probability": {
        "by_class": {
          "tia": 1.0,
          "mild_moderate_stroke": 0.4,
          "severe_stroke": 0.1
        }
      }
    },
    {
      "from": "Rehab Unit",
      "to": "Exit",
      "probability": 1.0
    }
  ]
}

## Validate with `ProcessModel`

In [3]:
model_instance = ProcessModel(**json_network)

In [4]:
print(model_instance)

ProcessModel(
    name='Stroke pathway with class-specific routing and reneging',
    description='Simple multi-class stroke pathway with class-specific routing, arrival, service, and reneging 
distributions.',
    customer_classes=[
        CustomerClass(name='tia', label='TIA'),
        CustomerClass(name='mild_moderate_stroke', label='Mild/moderate stroke'),
        CustomerClass(name='severe_stroke', label='Severe stroke')
    ],
    activities=[
        Activity(
            name='Acute Stroke Unit',
            type='activity',
            resource=Resource(name='Stroke beds', capacity=12),
            service_distribution=ClassDistributionMap(
                by_class={
                    'tia': Distribution(type='exponential', parameters={'mean': 2.0}),
                    'mild_moderate_stroke': Distribution(type='exponential', parameters={'mean': 4.0}),
                    'severe_stroke': Distribution(type='exponential', parameters={'mean': 6.0})
                }
            ),
            arrival_distribution=ClassDistributionMap(
                by_class={
                    'tia': Distribution(type='exponential', parameters={'rate': 2.0}),
                    'mild_moderate_stroke': Distribution(type='exponential', parameters={'rate': 1.2}),
                    'severe_stroke': Distribution(type='exponential', parameters={'rate': 0.6})
                }
            ),
            renege_distribution=ClassDistributionMap(
                by_class={
                    'tia': Distribution(type='uniform', parameters={'min': 2.0, 'max': 6.0}),
                    'mild_moderate_stroke': Distribution(type='uniform', parameters={'min': 4.0, 'max': 10.0}),
                    'severe_stroke': Distribution(type='uniform', parameters={'min': 6.0, 'max': 12.0})
                }
            )
        ),
        Activity(
            name='Rehab Unit',
            type='activity',
            resource=Resource(name='Rehab beds', capacity=8),
            service_distribution=ClassDistributionMap(
                by_class={
                    'mild_moderate_stroke': Distribution(type='exponential', parameters={'mean': 8.0}),
                    'severe_stroke': Distribution(type='exponential', parameters={'mean': 14.0})
                }
            ),
            arrival_distribution=None,
            renege_distribution=ClassDistributionMap(
                by_class={
                    'mild_moderate_stroke': Distribution(type='uniform', parameters={'min': 8.0, 'max': 16.0}),
                    'severe_stroke': Distribution(type='uniform', parameters={'min': 10.0, 'max': 20.0})
                }
            )
        )
    ],
    transitions=[
        Transition(
            source='Acute Stroke Unit',
            target='Rehab Unit',
            probability=ClassProbabilityMap(
                by_class={'tia': 0.0, 'mild_moderate_stroke': 0.6, 'severe_stroke': 0.9}
            )
        ),
        Transition(
            source='Acute Stroke Unit',
            target='Exit',
            probability=ClassProbabilityMap(
                by_class={'tia': 1.0, 'mild_moderate_stroke': 0.4, 'severe_stroke': 0.1}
            )
        ),
        Transition(source='Rehab Unit', target='Exit', probability=1.0)
    ]
)

In [5]:
model_instance.display_diagram(include_resources=False, show_class_arrivals=True)

```mermaid 
graph TD
    Arrivals_Acute_Stroke_Unit_tia("TIA</br>Time between arrivals<br/>Exponential(λ=2.0)")
    Arrivals_Acute_Stroke_Unit_mild_moderate_stroke("Mild/moderate stroke</br>Time between arrivals<br/>Exponential(λ=1.2)")
    Arrivals_Acute_Stroke_Unit_severe_stroke("Severe stroke</br>Time between arrivals<br/>Exponential(λ=0.6)")
    Acute_Stroke_Unit["Acute Stroke Unit</br>Class-specific service distributions (n=3)"]
    Rehab_Unit["Rehab Unit</br>Class-specific service distributions (n=2)"]
 Renege_Acute_Stroke_Unit{{"Renege</br>Class-specific reneging distributions (n=3)"}}
 Renege_Rehab_Unit{{"Renege</br>Class-specific reneging distributions (n=2)"}}
    Exit(["Exit"])

    Arrivals_Acute_Stroke_Unit_tia --> Acute_Stroke_Unit
    Arrivals_Acute_Stroke_Unit_mild_moderate_stroke --> Acute_Stroke_Unit
    Arrivals_Acute_Stroke_Unit_severe_stroke --> Acute_Stroke_Unit
    Acute_Stroke_Unit -.-> Renege_Acute_Stroke_Unit
    Rehab_Unit -.-> Renege_Rehab_Unit
 Acute_Stroke_Unit -->|TIA: 0%<br/>Mild/moderate stroke: 60%<br/>Severe stroke: 90%| Rehab_Unit
 Acute_Stroke_Unit -->|TIA: 100%<br/>Mild/moderate stroke: 40%<br/>Severe stroke: 10%| Exit
 Rehab_Unit --> Exit 
```

In [ ]:
model_instance.save_diagram("example7.mmd", include_resources=False)

In [6]:
model_instance.get_distributions_df()

,Activity,Phase,Customer Class,Customer Class Label,Distribution Type,Parameters
0,Acute Stroke Unit,Arrival,tia,TIA,Exponential,rate=2.0
1,Acute Stroke Unit,Arrival,mild_moderate_stroke,Mild/moderate stroke,Exponential,rate=1.2
2,Acute Stroke Unit,Arrival,severe_stroke,Severe stroke,Exponential,rate=0.6
3,Acute Stroke Unit,Service,tia,TIA,Exponential,mean=2.0
4,Acute Stroke Unit,Service,mild_moderate_stroke,Mild/moderate stroke,Exponential,mean=4.0
5,Acute Stroke Unit,Service,severe_stroke,Severe stroke,Exponential,mean=6.0
6,Acute Stroke Unit,Renege,tia,TIA,Uniform,"min=2.0, max=6.0"
7,Acute Stroke Unit,Renege,mild_moderate_stroke,Mild/moderate stroke,Uniform,"min=4.0, max=10.0"
8,Acute Stroke Unit,Renege,severe_stroke,Severe stroke,Uniform,"min=6.0, max=12.0"
9,Rehab Unit,Service,mild_moderate_stroke,Mild/moderate stroke,Exponential,mean=8.0


In [7]:
model_instance.get_routing_matrix_df()

Acute Stroke Unit  Rehab Unit  Exit
Customer Class       Source Activity                                       
tia                  Acute Stroke Unit                0.0         0.0   1.0
                     Rehab Unit                       0.0         0.0   1.0
mild_moderate_stroke Acute Stroke Unit                0.0         0.6   0.4
                     Rehab Unit                       0.0         0.0   1.0
severe_stroke        Acute Stroke Unit                0.0         0.9   0.1
                     Rehab Unit                       0.0         0.0   1.0

In [8]:
model_instance.get_resources_df()

,Resource,Activity,Count
0,Stroke beds,Acute Stroke Unit,12
1,Rehab beds,Rehab Unit,8


## Convert to `ciw` parameters

In [9]:
adapter = CiwConverter(model_instance)
network_params = adapter.generate_params()
print(network_params)

{
    'number_of_servers': [12, 8],
    'arrival_distributions': {
        'tia': [Exponential(rate=2.0), None],
        'mild_moderate_stroke': [Exponential(rate=1.2), None],
        'severe_stroke': [Exponential(rate=0.6), None]
    },
    'service_distributions': {
        'tia': [Exponential(rate=0.5), Deterministic(value=0.0)],
        'mild_moderate_stroke': [Exponential(rate=0.25), Exponential(rate=0.125)],
        'severe_stroke': [Exponential(rate=0.16666666666666666), Exponential(rate=0.07142857142857142)]
    },
    'routing': {
        'tia': [[0.0, 0.0], [0.0, 0.0]],
        'mild_moderate_stroke': [[0.0, 0.6], [0.0, 0.0]],
        'severe_stroke': [[0.0, 0.9], [0.0, 0.0]]
    },
    'reneging_time_distributions': {
        'tia': [Uniform(lower=2.0, upper=6.0), None],
        'mild_moderate_stroke': [Uniform(lower=4.0, upper=10.0), Uniform(lower=8.0, upper=16.0)],
        'severe_stroke': [Uniform(lower=6.0, upper=12.0), Uniform(lower=10.0, upper=20.0)]
    }
}

## Build and run the `ciw` model

In [10]:
network = ciw.create_network(**network_params)
sim = ciw.Simulation(network)
sim.simulate_until_max_time(50)
print("Quick simulation run worked!")

Quick simulation run worked!

## Run the model for multiple replications

In [11]:
df_reps = multiple_replications(
    network,
    model_instance,
    num_reps=5,
    runtime=2880,
    warmup=1440,
    n_jobs=-1,
)

df_reps.head()

,rep,node_id,activity_name,resource_name,resource_capacity,measure_scope,customer_class,n_service,mean_wait,mean_service,mean_Lq,utilisation,n_renege,renege_rate,mean_wait_renege,mean_wait_all
0,0,1,Acute Stroke Unit,Stroke beds,12,overall,All,4938,1.800132,3.351063,7.400534,96.463992,547,0.099727,3.231655,1.942893
1,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,tia,2356,1.606867,1.972439,3.718090,NaN,505,0.176512,3.105489,1.871391
2,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,mild_moderate_stroke,1673,1.935738,3.809967,2.387453,NaN,42,0.024490,4.748652,2.004625
3,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,severe_stroke,909,2.051470,6.079656,1.294990,NaN,0,0.000000,0.000000,2.051470
4,0,2,Rehab Unit,Rehab beds,8,overall,All,1024,11.212837,11.008924,14.102572,99.665328,764,0.427293,11.552040,11.357776


## Convert to wide format

In [12]:
# overall results
wide = tidy_to_wide_format(df_reps)
wide.head()

,mean_Lq [Acute Stroke Unit],mean_Lq [Rehab Unit],mean_service [Acute Stroke Unit],mean_service [Rehab Unit],mean_wait [Acute Stroke Unit],mean_wait [Rehab Unit],mean_wait_all [Acute Stroke Unit],mean_wait_all [Rehab Unit],mean_wait_renege [Acute Stroke Unit],mean_wait_renege [Rehab Unit],n_renege [Acute Stroke Unit],n_renege [Rehab Unit],n_service [Acute Stroke Unit],n_service [Rehab Unit],renege_rate [Acute Stroke Unit],renege_rate [Rehab Unit],utilisation [Acute Stroke Unit],utilisation [Rehab Unit]
rep,,,,,,,,,,,,,,,,,,
0,7.400534,14.102572,3.351063,11.008924,1.800132,11.212837,1.942893,11.357776,3.231655,11.552040,547,764,4938,1024,0.099727,0.427293,96.463992,99.665328
1,6.996143,13.648255,3.407212,11.046647,1.764208,10.931610,1.895831,11.072387,3.099027,11.258677,524,764,4790,1011,0.098607,0.430423,94.746258,99.508618
2,7.018241,14.552142,3.277717,11.384078,1.734655,11.299788,1.850287,11.345471,3.042278,11.398128,483,858,4979,989,0.088429,0.464537,94.738819,99.500339
3,7.531492,13.437849,3.433368,11.306926,1.866331,10.923390,1.999511,11.089114,3.107512,11.310522,582,747,4842,998,0.107301,0.428080,97.458552,99.680138
4,7.047902,13.743564,3.337193,11.607178,1.752768,11.179925,1.882578,11.251127,3.141273,11.338261,504,791,4887,968,0.093489,0.449687,95.763321,99.407975


In [13]:
# results by class = server strokes
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="severe_stroke")
wide_by_class.head()

,mean_Lq [Acute Stroke Unit],mean_Lq [Rehab Unit],mean_service [Acute Stroke Unit],mean_service [Rehab Unit],mean_wait [Acute Stroke Unit],mean_wait [Rehab Unit],mean_wait_all [Acute Stroke Unit],mean_wait_all [Rehab Unit],mean_wait_renege [Acute Stroke Unit],mean_wait_renege [Rehab Unit],n_renege [Acute Stroke Unit],n_renege [Rehab Unit],n_service [Acute Stroke Unit],n_service [Rehab Unit],renege_rate [Acute Stroke Unit],renege_rate [Rehab Unit]
rep,,,,,,,,,,,,,,,,
0,1.294990,6.818586,6.079656,13.605425,2.051470,11.799419,2.051470,12.077199,0.0,12.706388,0,249,909,564,0.0,0.306273
1,1.165035,6.173876,5.915932,13.474358,1.953028,11.375083,1.953028,11.636624,0.0,12.263161,0,225,859,539,0.0,0.294503
2,1.193721,6.611485,5.660921,13.692291,1.929246,11.720682,1.929246,11.990602,0.0,12.694849,0,220,891,574,0.0,0.277078
3,1.228988,6.356179,6.421085,14.217338,2.038875,11.418018,2.038875,11.734484,0.0,12.486606,0,231,868,549,0.0,0.296154
4,1.137005,6.118546,5.924261,14.874764,1.937617,11.533618,1.937617,11.826451,0.0,12.597816,0,205,845,540,0.0,0.275168


In [14]:
# results by class = tias
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="tia")
wide_by_class.head()

,mean_Lq [Acute Stroke Unit],mean_Lq [Rehab Unit],mean_service [Acute Stroke Unit],mean_service [Rehab Unit],mean_wait [Acute Stroke Unit],mean_wait [Rehab Unit],mean_wait_all [Acute Stroke Unit],mean_wait_all [Rehab Unit],mean_wait_renege [Acute Stroke Unit],mean_wait_renege [Rehab Unit],n_renege [Acute Stroke Unit],n_renege [Rehab Unit],n_service [Acute Stroke Unit],n_service [Rehab Unit],renege_rate [Acute Stroke Unit],renege_rate [Rehab Unit]
rep,,,,,,,,,,,,,,,,
0,3.718090,0.0,1.972439,0.0,1.606867,0.0,1.871391,0.0,3.105489,0.0,505,0,2356,0,0.176512,0.0
1,3.393180,0.0,2.060302,0.0,1.521546,0.0,1.790465,0.0,3.004134,0.0,495,0,2234,0,0.181385,0.0
2,3.500680,0.0,1.923626,0.0,1.557639,0.0,1.793943,0.0,2.985628,0.0,465,0,2345,0,0.165480,0.0
3,3.965209,0.0,1.989813,0.0,1.725705,0.0,1.977113,0.0,3.036294,0.0,554,0,2334,0,0.191828,0.0
4,3.602463,0.0,2.016627,0.0,1.592394,0.0,1.836299,0.0,3.049121,0.0,473,0,2352,0,0.167434,0.0


## Summarise results

In [15]:
df_reps.head(2)

,rep,node_id,activity_name,resource_name,resource_capacity,measure_scope,customer_class,n_service,mean_wait,mean_service,mean_Lq,utilisation,n_renege,renege_rate,mean_wait_renege,mean_wait_all
0,0,1,Acute Stroke Unit,Stroke beds,12,overall,All,4938,1.800132,3.351063,7.400534,96.463992,547,0.099727,3.231655,1.942893
1,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,tia,2356,1.606867,1.972439,3.718090,NaN,505,0.176512,3.105489,1.871391


In [16]:
summary = summarise_results(df_reps)
summary.round(1)

activity,Metric,Acute Stroke Unit (Stroke beds),Rehab Unit (Rehab beds)
0,Mean completed services,4887.2,998.0
1,Mean waiting time,1.8,11.1
2,Mean service time,3.4,11.3
3,Mean utilisation,95.8,99.6
4,Mean queue length,7.2,13.9
5,Mean reneges,528.0,784.8
6,Mean reneging rate,0.1,0.4
7,Mean reneging wait,3.1,11.4
8,Mean wait (all customers),1.9,11.2


In [17]:
summary_class = summarise_results_by_class(df_reps)
summary_class.round(1)

,Activity,Customer Class,Mean completed services,Mean waiting time,Mean service time,Mean queue length,Mean reneges,Mean reneging rate,Mean reneging wait,Mean wait (all customers)
0,Acute Stroke Unit,mild_moderate_stroke,1688.6,1.9,3.9,2.4,29.6,0.0,4.6,2.0
1,Acute Stroke Unit,severe_stroke,874.4,2.0,6.0,1.2,0.0,0.0,0.0,2.0
2,Acute Stroke Unit,tia,2324.2,1.6,2.0,3.6,498.4,0.2,3.0,1.9
3,Rehab Unit,mild_moderate_stroke,444.8,10.5,7.9,7.5,558.8,0.6,10.9,10.7
4,Rehab Unit,severe_stroke,553.2,11.6,14.0,6.4,226.0,0.3,12.5,11.9
5,Rehab Unit,tia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
